# Investigating the simplification of geometry
**Author**:  Sian Teesdale 

**Date created**:  14/04/2026

**Dataset Scope**: [one/many/all datasets]  

**Purpose**: The aim is to build upon 1_initial_analysis.ipynb, by seeing if other LPA data and other datasets are affected.

In [ ]:
import requests
import tempfile
import os

import pandas as pd
import geopandas as gpd


In [3]:
# Read in conservation-area
platform_ca = gpd.read_file("https://files.planning.data.gov.uk/dataset/conservation-area.geojson")
platform_lbo = gpd.read_file("https://files.planning.data.gov.uk/dataset/listed-building-outline.geojson")
platform_a4d = gpd.read_file("https://files.planning.data.gov.uk/dataset/article-4-direction-area.geojson")

# Read in LPA boundary
boundary = gpd.read_file("https://files.planning.data.gov.uk/dataset/local-planning-authority.geojson")


## Create util functions

In [ ]:
import shapely.wkt
import logging
from shapely import set_precision
from shapely.geometry import MultiPolygon
from shapely.geometry.polygon import orient
from shapely.validation import explain_validity, make_valid
from shapely.ops import transform
from pyproj import Transformer
from pyproj.transformer import TransformerGroup

osgb_to_wgs84 = Transformer.from_crs(27700, 4326, always_xy=True)


def dump_wkt(geometry, precision=6, dimensions=2):
    wkt = shapely.wkt.dumps(geometry, rounding_precision=precision, output_dimension=dimensions)
    return wkt.replace(", ", ",")


def make_multipolygon(geometry):
    # gap 1: guard for non-polygon types
    if geometry.geom_type in ["Point", "Line", "LineString", "MultiLineString"]:
        return None

    if geometry.geom_type == "MultiPolygon":
        return geometry

    if geometry.geom_type == "Polygon":
        return MultiPolygon([geometry])

    if geometry.geom_type == "GeometryCollection":
        polygons = []
        for geom in geometry.geoms:
            if geom.geom_type == "Polygon":
                polygons.append(geom)
            elif geom.geom_type == "MultiPolygon":
                for polygon in geom.geoms:
                    polygons.append(polygon)
            # gap 2: recursive handling of nested GeometryCollections
            elif geom.geom_type == "GeometryCollection":
                temp_polygons = make_multipolygon(geom)
                polygons.extend(temp_polygons.geoms)
            else:
                logging.info(f"skipping {geom.geom_type}")
        return MultiPolygon(polygons)

    raise ValueError(f"unexpected geometry {geometry.geom_type}")


def report(name, geom, prev=None, ref=None):
    try:
        coords = len(list(geom.geoms[0].exterior.coords))
    except AttributeError:
        coords = len(list(geom.exterior.coords))
    vs_raw_val = geom.symmetric_difference(ref).area * 111000 * 70000 if ref else 0.0
    vs_prev_val = geom.symmetric_difference(prev).area * 111000 * 70000 if prev else 0.0
    prev_str = f"{vs_prev_val:.4f} m²" if prev else "-"
    print(f"{name:<35} vertices={coords:>4}  vs_raw={vs_raw_val:.4f} m²  vs_prev={prev_str}")
    return vs_raw_val, vs_prev_val


def check_geometry(df, name):
    raw_df = df.loc[df['name'] == name]
    raw = raw_df.geometry.iloc[0]

    print(f"\n--- Checking geometry for '{name}' ---")

    steps = [("0. Raw", 0.0, 0.0)]

    report("0. Raw", raw)

    step2 = shapely.wkt.loads(dump_wkt(raw))
    steps.append(("1. After 6dp round-trip", *report("1. After 6dp round-trip", step2, raw, raw)))

    simplified = step2.simplify(0.000005)
    step3 = simplified if (not step2.is_valid or simplified.is_valid) else step2
    steps.append(("2. After simplify(0.000005)", *report("2. After simplify(0.000005)", step3, step2, raw)))

    step4 = set_precision(step3, 0.000001, mode="pointwise")
    steps.append(("3. After set_precision(1e-6)", *report("3. After set_precision(1e-6)", step4, step3, raw)))

    if not step4.is_valid:
        step5 = make_valid(step4)
        steps.append(("4. After make_valid", *report("4. After make_valid", step5, step4, raw)))
    else:
        step5 = step4
        print(f"{'4. make_valid':<35} skipped (geometry is valid)")

    step6 = make_multipolygon(step5)
    steps.append(("5. After make_multipolygon", *report("5. After make_multipolygon", step6, step5, raw)))

    if step6 and not step6.is_valid:
        step6b = step6.buffer(0)
        steps.append(("6. After buffer(0)", *report("6. After buffer(0)", step6b, step6, raw)))
    else:
        step6b = step6
        print(f"{'6. buffer(0)':<35} skipped (geometry is valid)")

    step6c = make_multipolygon(step6b)
    steps.append(("7. After 2nd make_multipolygon", *report("7. After 2nd make_multipolygon", step6c, step6b, raw)))

    polygons = [orient(g) for g in step6c.geoms]
    step7 = MultiPolygon(polygons)
    steps.append(("8. After orient", *report("8. After orient", step7, step6c, raw)))

    # Use vs_prev (not differences in vs_raw) to find the step with the greatest real change.
    # vs_raw can show spurious jumps at make_multipolygon because Shapely's symmetric_difference
    # between a MultiPolygon and a Polygon gives a non-zero result even when geometrically identical.
    # vs_prev compares consecutive geometry objects directly and correctly returns 0.0 for that step.
    real_increases = [(sname, svp) for sname, svr, svp in steps if svp > 0.01]
    if real_increases:
        biggest = max(real_increases, key=lambda x: x[1])
        print(f"\nGreatest increase in vs_raw: '{biggest[0]}' (+{biggest[1]:.4f} m²)")
    else:
        print("\nNo meaningful change in vs_raw across pipeline steps")


    print("\n--- Effect of varying precision ---")
    for dp in [6, 7, 8]:
        test = shapely.wkt.loads(dump_wkt(raw, precision=dp))
        test = test.simplify(0.000005)
        test = set_precision(test, 10**-dp, mode="pointwise")
        try:
            coords = len(list(test.geoms[0].exterior.coords))
        except AttributeError:
            coords = len(list(test.exterior.coords))
        diff = test.symmetric_difference(raw).area * 111000 * 70000
        print(f"  {dp}dp: vertices={coords}, diff vs raw={diff:.4f} m²")

    print("\n--- Effect of removing simplification ---")
    test_no_simplify = shapely.wkt.loads(dump_wkt(raw))
    test_no_simplify = set_precision(test_no_simplify, 0.000001, mode="pointwise")
    try:
        coords = len(list(test_no_simplify.geoms[0].exterior.coords))
    except AttributeError:
        coords = len(list(test_no_simplify.exterior.coords))
    diff = test_no_simplify.symmetric_difference(raw).area * 111000 * 70000
    print(f"  No simplify: vertices={coords}, diff vs raw={diff:.4f} m²")


## Conservation area - Barnet

This investigation came from Barnet, who highlighted they're seeing a difference in their raw data (endpoint) and our data. I've spotted this entity, through QGIS, that has the slightest variations that would be good to test.

- entity: 44002422
- entry-date: 20/03/2025 00:00:00 (GMT)
- name: Totteridge
- organisation-entity: 48
- prefix: conservation-area
- quality: authoritative
- reference: CA12
- start-date: 2008-05-06
- document-url: https://open.barnet.gov.uk/download/2nx73/evh
- documentation-url: https://www.barnet.gov.uk/node/9022


In [7]:
# Read in Barnet's data by bypassing their server block
url = "https://open.barnet.gov.uk/download/20yo8/c6n/conservation_area.gpkg"
headers = {"User-Agent": "Mozilla/5.0"}

with tempfile.NamedTemporaryFile(suffix=".gpkg", delete=False) as f:
    f.write(requests.get(url, headers=headers).content)
    tmp_path = f.name

barnet_data = gpd.read_file(tmp_path)
os.unlink(tmp_path)


---

Now we've confirmed the pipeline I'm running is the same as the platform data, let's see what step is actually producing the changes in geometry. My three culprits I want to check are:
1. 6 d.p.
2. Simplification
3. Precision

In [13]:
check_geometry(barnet_data, "Totteridge")


--- Checking geometry for 'Totteridge' ---
0. Raw                              vertices=2971  vs_raw=0.0000 m²  vs_prev=-
1. After 6dp round-trip             vertices=2971  vs_raw=224.4193 m²  vs_prev=224.4193 m²
2. After simplify(0.000005)         vertices= 520  vs_raw=1010.3070 m²  vs_prev=969.9118 m²
3. After set_precision(1e-6)        vertices= 520  vs_raw=857.9129 m²  vs_prev=0.0000 m²
4. make_valid                       skipped (geometry is valid)
5. After make_multipolygon          vertices= 520  vs_raw=1010.3070 m²  vs_prev=0.0000 m²
6. buffer(0)                        skipped (geometry is valid)
7. After 2nd make_multipolygon      vertices= 520  vs_raw=1010.3070 m²  vs_prev=0.0000 m²
8. After orient                     vertices= 520  vs_raw=1010.3070 m²  vs_prev=0.0000 m²

Greatest increase in vs_raw: '2. After simplify(0.000005)' (+785.8877 m²)

--- Effect of varying precision ---
  6dp: vertices=520, diff vs raw=857.9129 m²
  7dp: vertices=514, diff vs raw=1006.6178 m²
  8d

The simplification step is almost entirely responsible:

| Step | Vertices | vs Raw |
|------|----------|--------|
| Raw | 2971 | - |
| 6dp round-trip | 2971 | 224 m² |
| simplify(0.000005) | 520 | 1010 m² |
| set_precision(1e-6) | 520 | 858 m² |


Simplification removes 82% of vertices (2971 → 520) and causes ~1000 m² of change. Without simplification, the only difference from raw is 0.12 m² - essentially negligible rounding.

Interestingly, increasing decimal places (6 -> 7 -> 8) made the precision worse. Therefore 6 d.p. is acting as a pre-smoothing step.


---
Check other CA geometries to make sure they have the same effects

In [9]:
check_geometry(barnet_data, "Monken Hadley")


--- Checking geometry for 'Monken Hadley' ---
0. Raw                              vertices=2192  vs_raw=-  vs_prev=-
1. After 6dp round-trip             vertices=2192  vs_raw=282.4037 m²  vs_prev=282.4037 m²
2. After simplify(0.000005)         vertices= 468  vs_raw=1390.6041 m²  vs_prev=1344.0054 m²
3. After set_precision(1e-6)        vertices= 468  vs_raw=1200.3912 m²  vs_prev=0.0000 m²
4. make_valid                       skipped (geometry is valid)
5. After make_multipolygon          vertices= 468  vs_raw=1390.6041 m²  vs_prev=0.0000 m²
6. buffer(0)                        skipped (geometry is valid)
7. After 2nd make_multipolygon      vertices= 468  vs_raw=1390.6041 m²  vs_prev=0.0000 m²
8. After orient                     vertices= 468  vs_raw=1390.6041 m²  vs_prev=0.0000 m²

--- Effect of varying precision ---
  6dp: vertices=468, diff vs raw=1200.3912 m²
  7dp: vertices=476, diff vs raw=1317.8558 m²
  8dp: vertices=477, diff vs raw=1310.4371 m²

--- Effect of removing simplificat

In [10]:
check_geometry(barnet_data, "Wood Street")


--- Checking geometry for 'Wood Street' ---
0. Raw                              vertices=1196  vs_raw=-  vs_prev=-
1. After 6dp round-trip             vertices=1196  vs_raw=89.1774 m²  vs_prev=89.1774 m²
2. After simplify(0.000005)         vertices= 271  vs_raw=365.7166 m²  vs_prev=348.5242 m²
3. After set_precision(1e-6)        vertices= 271  vs_raw=298.2786 m²  vs_prev=0.0000 m²
4. make_valid                       skipped (geometry is valid)
5. After make_multipolygon          vertices= 271  vs_raw=365.7166 m²  vs_prev=0.0000 m²
6. buffer(0)                        skipped (geometry is valid)
7. After 2nd make_multipolygon      vertices= 271  vs_raw=365.7166 m²  vs_prev=0.0000 m²
8. After orient                     vertices= 271  vs_raw=365.7166 m²  vs_prev=0.0000 m²

--- Effect of varying precision ---
  6dp: vertices=271, diff vs raw=298.2786 m²
  7dp: vertices=266, diff vs raw=348.6700 m²
  8dp: vertices=261, diff vs raw=387.0178 m²

--- Effect of removing simplification ---
  No 

## Listed building outline - Barking and Dagenham

In [27]:
url = "https://services3.arcgis.com/lCzPKKaGs7lhrnrV/arcgis/rest/services/Listed_Building_Area_Data/FeatureServer/0/query?where=1%3D1&outFields=*&f=geojson"
barking_data = gpd.read_file(url)
print(len(barking_data))

48


In [28]:
# Fix column names
barking_data.columns = barking_data.columns.str.lower()
cols = list(barking_data.columns)
cols[3] = 'geometry_type'
barking_data.columns = cols

In [32]:
barking_data.head(2)

,objectid,name,reference,geometry_type,document_url,listed_building,listed_building_grade,notes,start_date,end_date,entry_date,shape__area,shape__length,geometry
0,1,VALENCE HOUSE,LB001,Polygon,https://www.lbbd.gov.uk/planning-building-cont...,1064404,II*,https://historicengland.org.uk/listing/the-lis...,28/06/1954,None,28/06/1954,372.612259,119.966124,"POLYGON ((0.13416 51.55827, 0.13412 51.55818, ..."
1,2,FURZE HOUSE FARMHOUSE,LB002,Polygon,https://www.lbbd.gov.uk/planning-building-cont...,1064405,II,https://historicengland.org.uk/listing/the-lis...,24/08/1981,None,24/08/1981,114.183327,50.805413,"POLYGON ((0.13925 51.59184, 0.13924 51.59184, ..."


Let's look at Valence House - entity 42120505. https://www.planning.data.gov.uk/entity/42120505

In [30]:
check_geometry(barking_data, "VALENCE HOUSE")


--- Checking geometry for 'VALENCE HOUSE' ---
0. Raw                              vertices=  19  vs_raw=0.0000 m²  vs_prev=-
1. After 6dp round-trip             vertices=  19  vs_raw=1.9508 m²  vs_prev=1.9508 m²
2. After simplify(0.000005)         vertices=  19  vs_raw=1.9508 m²  vs_prev=0.0000 m²
3. After set_precision(1e-6)        vertices=  19  vs_raw=0.0000 m²  vs_prev=0.0000 m²
4. make_valid                       skipped (geometry is valid)
5. After make_multipolygon          vertices=  19  vs_raw=1.9508 m²  vs_prev=0.0000 m²
6. buffer(0)                        skipped (geometry is valid)
7. After 2nd make_multipolygon      vertices=  19  vs_raw=1.9508 m²  vs_prev=0.0000 m²
8. After orient                     vertices=  19  vs_raw=1.9508 m²  vs_prev=0.0000 m²

Greatest increase in vs_raw: '1. After 6dp round-trip' (+1.9508 m²)

--- Effect of varying precision ---
  6dp: vertices=19, diff vs raw=0.0000 m²
  7dp: vertices=19, diff vs raw=0.0000 m²
  8dp: vertices=19, diff vs raw=0.

This tells a completely different story from Barnet. Key observations:
- Only 19 vertices... this is already a very simple geometry, so simplification has nothing to remove. The pipeline was designed for exactly this kind of data.
- 6dp round-trip causes the only change (1.95 m²), but then set_precision corrects it back to 0. This happens because Valence House's coordinates already have ≤6dp precision, so snapping to a 1e-6 grid undoes the rounding rather than compounding it.
- Varying precision all give 0 m² — confirms the source data is already at 6dp or coarser, so there's no extra precision being lost.

This contrasts sharply with Barnet. The pipeline works well for simple, low-precision geometries like Barking's. The problem is specifically with high-detail WGS84 sources like Barnet's, where the simplification step aggressively removes detail that doesn't need removing.

One thing worth noting... the vs_raw jumping back to 1.95 m² at step 5 (make_multipolygon) after step 3 showed 0.00 m² looks like a Shapely floating-point quirk when comparing a MultiPolygon against a Polygon via symmetric_difference. It's not a real geometric difference.

---
Check another LBO - Furze House Farmhouse (42120506) https://www.planning.data.gov.uk/entity/42120506

In [33]:
check_geometry(barking_data, "FURZE HOUSE FARMHOUSE")


--- Checking geometry for 'FURZE HOUSE FARMHOUSE' ---
0. Raw                              vertices=  11  vs_raw=0.0000 m²  vs_prev=-
1. After 6dp round-trip             vertices=  11  vs_raw=0.9879 m²  vs_prev=0.9879 m²
2. After simplify(0.000005)         vertices=  11  vs_raw=0.9879 m²  vs_prev=0.0000 m²
3. After set_precision(1e-6)        vertices=  11  vs_raw=0.0000 m²  vs_prev=0.0000 m²
4. make_valid                       skipped (geometry is valid)
5. After make_multipolygon          vertices=  11  vs_raw=0.9879 m²  vs_prev=0.0000 m²
6. buffer(0)                        skipped (geometry is valid)
7. After 2nd make_multipolygon      vertices=  11  vs_raw=0.9879 m²  vs_prev=0.0000 m²
8. After orient                     vertices=  11  vs_raw=0.9879 m²  vs_prev=0.0000 m²

Greatest increase in vs_raw: '1. After 6dp round-trip' (+0.9879 m²)

--- Effect of varying precision ---
  6dp: vertices=11, diff vs raw=0.0000 m²
  7dp: vertices=11, diff vs raw=0.0000 m²
  8dp: vertices=11, diff v

Same pattern observed

## Article 4 direction - Liverpool

In [50]:
lpool_data = pd.read_csv('https://liverpool.gov.uk/media/ilojkmgw/pd-rights-removed.csv')

# Rename columns and convert to GeoDataFrame
lpool_data.rename(columns={'WKT': 'geometry'}, inplace=True)
lpool_data = gpd.GeoDataFrame(lpool_data)

# Parse WKT strings into Shapely geometries and set CRS
lpool_data['geometry'] = lpool_data['geometry'].apply(shapely.wkt.loads)
lpool_data = gpd.GeoDataFrame(lpool_data, geometry='geometry', crs='EPSG:27700')

# Reproject to WGS84 (as the pipeline would do)
lpool_data = lpool_data.to_crs('EPSG:4326')

In [52]:
lpool_data.head(2)

,geometry,fid,reference,article-4-direction,name,address-text,permitted-development-rights,notes,start-date,entry-date,end-date,documentation-url,document-url
0,"MULTIPOLYGON (((-2.91926 53.42053, -2.91926 53...",1,A4Da407,A4D407,"The Liverpool (Anfield, Central, Greenbank, Ke...","Entire Wards: Anfield, Central, Greenbank, Ken...",3L(b),"The Liverpool (Anfield, Central, Greenbank, Ke...",10/04/2021,10/04/2021,NaN,https://liverpool.gov.uk/media/2ofdsdbj/11-war...,https://liverpool.gov.uk/media/2ofdsdbj/11-war...
1,"MULTIPOLYGON (((-2.93058 53.39055, -2.93059 53...",2,A4Da408,A4D408,The Dales,"Gainsborough Road, Odds Nos 3 to 95A, Evens No...",3L(b),The Dales (Article 4(1)) Direction 2018,19/07/2018,19/07/2018,NaN,https://liverpool.gov.uk/media/euvnr0wf/art-4-...,https://liverpool.gov.uk/media/euvnr0wf/art-4-...


Let's check The Liverpool [...] (7010007582) A4D area first (https://www.planning.data.gov.uk/entity/7010007582)

In [54]:
check_geometry(lpool_data, "The Liverpool (Anfield, Central, Greenbank, Kensington and Fairfield, Picton, Princess Park, Riverside, Tuebrook and Stoneycroft and Wavertree wards and parts of Kirkdale and Church wards) (Article 4(1)) Direction 2020")


--- Checking geometry for 'The Liverpool (Anfield, Central, Greenbank, Kensington and Fairfield, Picton, Princess Park, Riverside, Tuebrook and Stoneycroft and Wavertree wards and parts of Kirkdale and Church wards) (Article 4(1)) Direction 2020' ---
0. Raw                              vertices=1230  vs_raw=0.0000 m²  vs_prev=-
1. After 6dp round-trip             vertices=1230  vs_raw=642.8605 m²  vs_prev=642.8605 m²
2. After simplify(0.000005)         vertices= 703  vs_raw=2183.3962 m²  vs_prev=1805.1794 m²
3. After set_precision(1e-6)        vertices= 703  vs_raw=1665.2625 m²  vs_prev=0.0000 m²
4. make_valid                       skipped (geometry is valid)
5. After make_multipolygon          vertices= 703  vs_raw=2183.3962 m²  vs_prev=0.0000 m²
6. buffer(0)                        skipped (geometry is valid)
7. After 2nd make_multipolygon      vertices= 703  vs_raw=2183.3962 m²  vs_prev=0.0000 m²
8. After orient                     vertices= 703  vs_raw=2183.3962 m²  vs_prev=0.0000 

Liverpool's raw data was initially in WKT strings, and in OSGB CRS (i.e. northings and eastings). The results above show a very slightly different, but still similar, story to Barnet.

Key observations:
- This A4D area is huge, and therefore sees a big change in area and vertices during the 6 d.p. and simplification step
- Liverpool's 6dp round-trip is much larger than Barnet's (643 m² vs 224 m²) — because OSGB→WGS84 conversion produces coordinates with many decimal places, so the 6dp truncation cuts off more information than with native WGS84 data
- Simplification is the dominant culprit for both complex geometries regardless of source CRS
- Without simplification, all datasets land at ~0.1 m² — confirming the finding holds broadly, not just for Barnet

---
Check with 'The Dales' - entity 7010007583 - too (https://www.planning.data.gov.uk/entity/7010007583)

In [55]:
check_geometry(lpool_data, "The Dales")


--- Checking geometry for 'The Dales' ---
0. Raw                              vertices=  73  vs_raw=0.0000 m²  vs_prev=-
1. After 6dp round-trip             vertices=  73  vs_raw=33.3165 m²  vs_prev=33.3165 m²
2. After simplify(0.000005)         vertices=  40  vs_raw=118.1341 m²  vs_prev=93.3909 m²
3. After set_precision(1e-6)        vertices=  40  vs_raw=82.8010 m²  vs_prev=0.0000 m²
4. make_valid                       skipped (geometry is valid)
5. After make_multipolygon          vertices=  40  vs_raw=118.1341 m²  vs_prev=0.0000 m²
6. buffer(0)                        skipped (geometry is valid)
7. After 2nd make_multipolygon      vertices=  40  vs_raw=118.1341 m²  vs_prev=0.0000 m²
8. After orient                     vertices=  40  vs_raw=118.1341 m²  vs_prev=0.0000 m²

Greatest increase in vs_raw: '2. After simplify(0.000005)' (+84.8176 m²)

--- Effect of varying precision ---
  6dp: vertices=40, diff vs raw=82.8010 m²
  7dp: vertices=38, diff vs raw=130.7704 m²
  8dp: vertices=38

## Tree Preservation Orders - 

In [62]:
url = "https://mapservices.leeds.gov.uk/arcgis/rest/services/Public/Strategic_Planning/MapServer/61"

# Check record count and server limit first
info = requests.get(url + "?f=json").json()
count = requests.get(url + "/query?where=1%3D1&returnCountOnly=true&f=json").json()
print(f"Max record count: {info.get('maxRecordCount')}")
print(f"Total features:   {count.get('count')}")

# If total <= maxRecordCount, read directly
query_url = url + "/query?where=1%3D1&outFields=*&f=geojson"
leeds_data = gpd.read_file(query_url)
print(leeds_data.crs)
print(len(leeds_data))

Max record count: 5000
Total features:   3708
EPSG:4326
3708


In [64]:
leeds_data.columns = leeds_data.columns.str.lower()

In [ ]:
leeds_data.head(2)

# Looks like there's repeated names - need to find a unique one to check

,objectid,reference,name,tree_preservation_order,tree_preservation_zone_type,start_date,entry_date,end_date,tpobject_species,tpobject_count,shape.area,shape.len,geometry
0,433,TPO1999_050_G_001,G1,TPO1999_050,Group,1999-10-21,1999-10-21,None,Not recorded,Not recorded,162.651967,52.617389,"POLYGON ((-1.62652 53.8433, -1.62655 53.84331,..."
1,434,TPO1989_005_G_001,G1,TPO1989_005,Group,1989-02-07,1989-02-07,None,Not recorded,Not recorded,1335.224768,228.527358,"POLYGON ((-1.58029 53.8476, -1.58031 53.8476, ..."


In [67]:
leeds_data['name'].value_counts()

name
G1     999
G2     505
G3     319
A1     295
W1     248
      ... 
W24      1
A21      1
A39      1
A25      1
A26      1
Name: count, Length: 108, dtype: int64

In [72]:
check_geometry(leeds_data, "W24") # https://www.planning.data.gov.uk/entity/19169780


--- Checking geometry for 'W24' ---
0. Raw                              vertices= 243  vs_raw=0.0000 m²  vs_prev=-
1. After 6dp round-trip             vertices= 243  vs_raw=21.7290 m²  vs_prev=21.7290 m²
2. After simplify(0.000005)         vertices=  42  vs_raw=65.0078 m²  vs_prev=57.7232 m²
3. After set_precision(1e-6)        vertices=  42  vs_raw=50.6682 m²  vs_prev=0.0000 m²
4. make_valid                       skipped (geometry is valid)
5. After make_multipolygon          vertices=  42  vs_raw=65.0078 m²  vs_prev=0.0000 m²
6. buffer(0)                        skipped (geometry is valid)
7. After 2nd make_multipolygon      vertices=  42  vs_raw=65.0078 m²  vs_prev=0.0000 m²
8. After orient                     vertices=  42  vs_raw=65.0078 m²  vs_prev=0.0000 m²

Greatest increase in vs_raw: '2. After simplify(0.000005)' (+43.2788 m²)

--- Effect of varying precision ---
  6dp: vertices=42, diff vs raw=50.6682 m²
  7dp: vertices=41, diff vs raw=55.6653 m²
  8dp: vertices=41, diff vs r

In [71]:
check_geometry(leeds_data, "A26") # https://www.planning.data.gov.uk/entity/19171337


--- Checking geometry for 'A26' ---
0. Raw                              vertices=  11  vs_raw=0.0000 m²  vs_prev=-
1. After 6dp round-trip             vertices=  11  vs_raw=9.9955 m²  vs_prev=9.9955 m²
2. After simplify(0.000005)         vertices=  11  vs_raw=9.9955 m²  vs_prev=0.0000 m²
3. After set_precision(1e-6)        vertices=  11  vs_raw=0.0000 m²  vs_prev=0.0000 m²
4. make_valid                       skipped (geometry is valid)
5. After make_multipolygon          vertices=  11  vs_raw=9.9955 m²  vs_prev=0.0000 m²
6. buffer(0)                        skipped (geometry is valid)
7. After 2nd make_multipolygon      vertices=  11  vs_raw=9.9955 m²  vs_prev=0.0000 m²
8. After orient                     vertices=  11  vs_raw=9.9955 m²  vs_prev=0.0000 m²

Greatest increase in vs_raw: '5. After make_multipolygon' (+9.9955 m²)

--- Effect of varying precision ---
  6dp: vertices=11, diff vs raw=0.0000 m²
  7dp: vertices=11, diff vs raw=0.0000 m²
  8dp: vertices=11, diff vs raw=0.0000 m²